# Notebook 12 — Mini-Project and Assessment

**Module:** 14 — Deep Learning and GNNs  
**Tier:** 2 — Working competence  
**Notebook:** 12 of 12  
**Time estimate:** 4–6 hours total (50 points)

> This notebook has two parts:
> - **MP01:** DeepBind CNN vs. k-mer+SVM — a reproduction and comparison study
> - **Assessment:** 5 questions, 10 points each, covering the theoretical and
>   practical content from the full Module 14 sequence.
>
> Exercise solutions belong in `exercises/`. Do not write answers inline here.

---
## MP01 — TF Binding Classification: DeepBind CNN vs. k-mer + SVM

**Goal:** reproduce the DeepBind benchmark result on a synthetic dataset, then compare
against the k-mer + SVM approach from Module 13 NB13.

**This is direct interview practice.** The question "how does DeepBind compare to
k-mer models" appears routinely in bioinformatics computational biology interviews.
You need a quantitative answer (AUROC gap) and a mechanistic explanation (why CNNs
outperform bag-of-words features on motif discovery).

**Dataset design:**
- N=2000 sequences, length=100bp, binary labels (TF binding: yes/no)
- Positive: contains motif with up to 10% noise substitutions
- Negative: random background sequences
- 60/20/20 train/val/test split
- Motif: GATAAG (GATA factor binding site)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
from sklearn.model_selection import train_test_split
from itertools import product

torch.manual_seed(42); np.random.seed(42)
rng = np.random.default_rng(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Constants ----
NUCL = 'ACGT'
NUCL_IDX = {b: i for i, b in enumerate(NUCL)}
MOTIF = 'GATAAG'
SEQ_L = 100
N = 2000

# ---- Data generation ----
def plant_motif(seq_list, motif, noise=0.10):
    """Plant motif in a random position with up to noise-fraction substitutions."""
    seqs = []
    for seq in seq_list:
        pos = rng.integers(0, SEQ_L - len(motif))
        s = list(seq)
        for j, b in enumerate(motif):
            if rng.random() > noise:
                s[pos + j] = b
            else:
                s[pos + j] = rng.choice(list(NUCL))
        seqs.append(''.join(s))
    return seqs

def random_seqs(n):
    return [''.join(rng.choice(list(NUCL), SEQ_L)) for _ in range(n)]

pos_raw = random_seqs(N // 2)
pos_seqs = plant_motif(pos_raw, MOTIF, noise=0.10)
neg_seqs = random_seqs(N // 2)
all_seqs = pos_seqs + neg_seqs
all_y = np.array([1]*(N//2) + [0]*(N//2), dtype=np.float32)

# 60/20/20 split
idx = np.arange(N)
idx_tr, idx_tmp = train_test_split(idx, test_size=0.40, stratify=all_y, random_state=42)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, stratify=all_y[idx_tmp], random_state=42)

print(f'Split: train={len(idx_tr)}, val={len(idx_va)}, test={len(idx_te)}')
print(f'Positive rate (train): {all_y[idx_tr].mean():.2f}')


In [ ]:
# ---- One-hot encoding ----
def one_hot_encode(seqs):
    X = np.zeros((len(seqs), 4, SEQ_L), dtype=np.float32)
    for i, seq in enumerate(seqs):
        for j, b in enumerate(seq):
            if b in NUCL_IDX: X[i, NUCL_IDX[b], j] = 1.0
    return X

X_oh = one_hot_encode(all_seqs)  # (N, 4, L)

# ---- k-mer features (k=4) ----
def kmer_features(seqs, k=4):
    kmers = [''.join(p) for p in product(NUCL, repeat=k)]
    kmer_idx = {km: i for i, km in enumerate(kmers)}
    X = np.zeros((len(seqs), len(kmers)), dtype=np.float32)
    for i, seq in enumerate(seqs):
        for j in range(len(seq) - k + 1):
            km = seq[j:j+k]
            if km in kmer_idx:
                X[i, kmer_idx[km]] += 1
        X[i] /= max(len(seq) - k + 1, 1)
    return X

print('Computing k-mer features (k=4)...')
X_kmer = kmer_features(all_seqs, k=4)  # (N, 256)
print(f'k-mer feature matrix: {X_kmer.shape}')

# ---- Datasets ----
class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_ds = SeqDS(X_oh[idx_tr], all_y[idx_tr])
val_ds   = SeqDS(X_oh[idx_va], all_y[idx_va])
test_ds  = SeqDS(X_oh[idx_te], all_y[idx_te])

train_ld = DataLoader(train_ds, batch_size=64, shuffle=True)
val_ld   = DataLoader(val_ds, batch_size=400)
test_ld  = DataLoader(test_ds, batch_size=400)


In [ ]:
# ---- DeepBind CNN ----
class DeepBind(nn.Module):
    def __init__(self, n_filters=32, kernel_size=7, fc_units=64, dropout=0.3):
        super().__init__()
        self.conv = nn.Conv1d(4, n_filters, kernel_size)
        self.bn = nn.BatchNorm1d(n_filters)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(n_filters, fc_units)
        self.fc2 = nn.Linear(fc_units, 1)

    def forward(self, x):
        h = self.pool(torch.relu(self.bn(self.conv(x)))).squeeze(-1)
        h = self.dropout(torch.relu(self.fc1(h)))
        return torch.sigmoid(self.fc2(h))

    def get_filter_weights(self):
        """Return conv filters (n_filters, 4, kernel_size) for motif visualization."""
        return self.conv.weight.data.cpu().numpy()

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    losses = []
    for X_b, y_b in loader:
        optimizer.zero_grad()
        pred = model(X_b.to(device))
        loss = criterion(pred, y_b.to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)

def evaluate(model, loader):
    model.eval()
    y_probs, y_true = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            y_probs.extend(model(X_b.to(device)).cpu().numpy().ravel())
            y_true.extend(y_b.numpy().ravel())
    return np.array(y_true), np.array(y_probs)

cnn_model = DeepBind(n_filters=32, kernel_size=7, fc_units=64, dropout=0.3).to(device)
cnn_opt = optim.Adam(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(cnn_opt, T_max=60, eta_min=1e-5)
criterion = nn.BCELoss()

best_auroc, best_state = 0, None
cnn_train_losses, cnn_val_aurocs = [], []
for epoch in range(60):
    tl = train_epoch(cnn_model, train_ld, criterion, cnn_opt)
    y_v, p_v = evaluate(cnn_model, val_ld)
    auroc = roc_auc_score(y_v, p_v)
    scheduler.step()
    cnn_train_losses.append(tl)
    cnn_val_aurocs.append(auroc)
    if auroc > best_auroc:
        best_auroc = auroc
        import copy; best_state = copy.deepcopy(cnn_model.state_dict())

cnn_model.load_state_dict(best_state)
y_test_true, cnn_test_probs = evaluate(cnn_model, test_ld)
cnn_auroc = roc_auc_score(y_test_true, cnn_test_probs)
cnn_auprc = average_precision_score(y_test_true, cnn_test_probs)
print(f'CNN (DeepBind) test AUROC: {cnn_auroc:.4f}, AUPRC: {cnn_auprc:.4f}')


In [ ]:
# ---- k-mer + SVM baseline ----
scaler_kmer = StandardScaler()
X_kmer_tr_s = scaler_kmer.fit_transform(X_kmer[idx_tr])
X_kmer_va_s = scaler_kmer.transform(X_kmer[idx_va])
X_kmer_te_s = scaler_kmer.transform(X_kmer[idx_te])

svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm_model.fit(X_kmer_tr_s, all_y[idx_tr])

svm_val_probs = svm_model.predict_proba(X_kmer_va_s)[:, 1]
svm_test_probs = svm_model.predict_proba(X_kmer_te_s)[:, 1]
svm_auroc = roc_auc_score(y_test_true, svm_test_probs)
svm_auprc = average_precision_score(y_test_true, svm_test_probs)
print(f'k-mer (k=4) + SVM test AUROC: {svm_auroc:.4f}, AUPRC: {svm_auprc:.4f}')
print(f'\nDeepBind AUROC advantage: {cnn_auroc - svm_auroc:+.4f}')


In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Panel A: ROC curves
ax = axes[0, 0]
for probs, name, color in [
        (cnn_test_probs, f'DeepBind CNN (AUROC={cnn_auroc:.3f})', 'steelblue'),
        (svm_test_probs, f'k-mer+SVM   (AUROC={svm_auroc:.3f})', 'tomato')]:
    fpr, tpr, _ = roc_curve(y_test_true, probs)
    ax.plot(fpr, tpr, color=color, lw=2, label=name)
ax.plot([0,1],[0,1], 'k--', lw=0.8)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('A. ROC curves (test set)')
ax.legend(fontsize=9)

# Panel B: Training curve (CNN)
ax = axes[0, 1]
ax_r = ax.twinx()
ax.plot(cnn_train_losses, 'grey', lw=1.5, label='Train loss')
ax_r.plot(cnn_val_aurocs, 'steelblue', lw=1.5, label='Val AUROC')
ax_r.axhline(cnn_auroc, color='steelblue', ls='--', lw=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss', color='grey')
ax_r.set_ylabel('AUROC', color='steelblue')
ax.set_title('B. CNN training curve')
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax_r.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, fontsize=8)

# Panel C: Learned CNN filters (top 4 by max weight)
ax = axes[0, 2]
filters = cnn_model.get_filter_weights()  # (32, 4, 7)
max_weights = np.abs(filters).max(axis=(1,2))
top_filters = np.argsort(max_weights)[::-1][:4]
vmin = filters[top_filters].min(); vmax = filters[top_filters].max()
for fi, f_idx in enumerate(top_filters):
    im = ax.imshow(filters[f_idx], aspect='auto', cmap='RdBu_r',
                     vmin=vmin, vmax=vmax,
                     extent=[0, filters.shape[-1], fi-0.4+0, fi+0.4])
ax.set_yticks(range(4))
ax.set_yticklabels([f'Filter {top_filters[i]}' for i in range(4)], fontsize=8)
ax.set_xticks(range(filters.shape[-1]))
ax.set_xlabel('Position'); ax.set_title('C. Top-4 CNN filters\n(learned motif detectors)')
plt.colorbar(im, ax=ax, shrink=0.6)

# Panel D: AUROC + AUPRC comparison bar chart
ax = axes[1, 0]
metrics = {
    'AUROC': (cnn_auroc, svm_auroc),
    'AUPRC': (cnn_auprc, svm_auprc)
}
x = np.arange(2); w = 0.35
ax.bar(x - w/2, [cnn_auroc, cnn_auprc], w, label='DeepBind CNN', color='steelblue', edgecolor='black')
ax.bar(x + w/2, [svm_auroc, svm_auprc], w, label='k-mer+SVM', color='tomato', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(['AUROC', 'AUPRC'])
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('D. CNN vs SVM: AUROC and AUPRC\n(test set summary)')
ax.legend(fontsize=9)
for i, (cv, sv) in enumerate([(cnn_auroc, svm_auroc), (cnn_auprc, svm_auprc)]):
    ax.text(i-w/2, cv+0.01, f'{cv:.3f}', ha='center', fontsize=8)
    ax.text(i+w/2, sv+0.01, f'{sv:.3f}', ha='center', fontsize=8)

# Panel E: Motif logo from best CNN filter
ax = axes[1, 1]
best_filter = filters[top_filters[0]]  # (4, kernel_size)
# Convert to softmax probability (PWM interpretation)
filt_softmax = np.exp(best_filter) / np.exp(best_filter).sum(axis=0, keepdims=True)
for j in range(filt_softmax.shape[1]):
    ht = 0
    sorted_aa = np.argsort(filt_softmax[:, j])
    for aa in sorted_aa:
        prob = filt_softmax[aa, j]
        ax.bar(j, prob, bottom=ht, color=['#4daf4a','#377eb8','#e41a1c','#ff7f00'][aa],
               edgecolor='black', lw=0.3, label=NUCL[aa] if j==0 else '')
        ht += prob
ax.set_xticks(range(filt_softmax.shape[1]))
ax.set_xticklabels([f'P{i+1}' for i in range(filt_softmax.shape[1])])
ax.set_ylabel('Probability'); ax.set_title(f'E. Best CNN filter (motif logo)\nTrue motif: {MOTIF}')
ax.set_ylim(0, 1.05)
handles = [plt.Rectangle((0,0),1,1, color=['#4daf4a','#377eb8','#e41a1c','#ff7f00'][i]) for i in range(4)]
ax.legend(handles, list(NUCL), fontsize=8, loc='upper right')

# Panel F: Interpretation summary
ax = axes[1, 2]
ax.axis('off')
text = (
    'MP01 — Interpretation summary\n\n'
    f'DeepBind CNN\n'
    f'  AUROC: {cnn_auroc:.3f}\n'
    f'  AUPRC: {cnn_auprc:.3f}\n\n'
    f'k-mer (k=4) + SVM\n'
    f'  AUROC: {svm_auroc:.3f}\n'
    f'  AUPRC: {svm_auprc:.3f}\n\n'
    'WHY does CNN win?\n'
    '• k-mer model treats motif as bag-of-words.\n'
    '• Cannot distinguish GATAA in position 5\n'
    '  from GATAA in position 85.\n'
    '• CNN conv filter scores the exact\n'
    '  local motif pattern at any position\n'
    '  (position-invariant by global max pool).\n'
    '• Positional bag-of-kmers overlaps signal\n'
    '  with spurious co-occurring sequences.'
)
ax.text(0.05, 0.95, text, transform=ax.transAxes, fontsize=9,
          verticalalignment='top', fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Module 14 NB12 MP01: DeepBind CNN vs. k-mer+SVM\nTF Binding Classification Benchmark', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('mini_project_mp01.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Assessment (50 points)

**Instructions:**
- Write all answers in `exercises/12_assessment_answers.md`
- Each question is 10 points (partial credit given for partially correct reasoning)
- Mathematical derivations should show intermediate steps
- Code snippets should be fully runnable

---

### Q1 — Backpropagation derivation (10 points)

Consider a 2-layer neural network:
- Layer 1: $z^{(1)} = W^{(1)} x + b^{(1)}$, $a^{(1)} = \text{ReLU}(z^{(1)})$
- Layer 2: $z^{(2)} = W^{(2)} a^{(1)} + b^{(2)}$, $\hat{y} = \sigma(z^{(2)})$
- Loss: binary cross-entropy $\mathcal{L} = -y \log \hat{y} - (1-y)\log(1-\hat{y})$

**(a)** Derive $\partial \mathcal{L} / \partial W^{(2)}$ using the chain rule. Show every step.

**(b)** Derive $\partial \mathcal{L} / \partial W^{(1)}$ using the chain rule. Where does the
ReLU derivative appear and what is it?

**(c)** What happens to $\partial \mathcal{L} / \partial W^{(1)}$ for a unit where $z^{(1)}_j < 0$?
Why is this called the "dying ReLU" problem?

---

### Q2 — GCN update rule (10 points)

The Graph Convolutional Network update rule is:
$$H^{(l+1)} = \sigma\left(\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} H^{(l)} W^{(l)}\right)$$

where $\tilde{A} = A + I$ and $\tilde{D}_{ii} = \sum_j \tilde{A}_{ij}$.

**(a)** Why is the identity matrix $I$ added to $A$ before normalization? What biological
operation does this correspond to?

**(b)** Write out explicitly what the update $H^{(l+1)}_{v}$ computes for node $v$ — describe
it in words (not just formula): where does node $v$'s new representation come from?

**(c)** Implement the normalized adjacency computation from scratch (without PyTorch Geometric)
for the graph: nodes {0,1,2,3}, edges {(0,1), (1,2), (2,3), (0,3)}.
Show the full $4 \times 4$ normalized adjacency matrix.

---

### Q3 — VAE ELBO (10 points)

The VAE is trained by maximizing the Evidence Lower BOund:
$$\mathcal{L}_{ELBO}(x) = \mathbb{E}_{q(z|x)}[\log p(x|z)] - D_{KL}(q(z|x) \| p(z))$$

**(a)** What does each term represent biologically when applied to single-cell RNA-seq data?
Describe the prior $p(z)$, the approximate posterior $q(z|x)$, and the likelihood $p(x|z)$
in biological terms (what are $x$ and $z$?).

**(b)** Write out the closed-form KL divergence for $q(z|x) = \mathcal{N}(\mu, \text{diag}(\sigma^2))$
vs. $p(z) = \mathcal{N}(0, I)$. Show the derivation.

**(c)** What is the reparameterization trick and why is it necessary for training via gradient
descent? Write the reparameterized sample $z$ in terms of $\mu$, $\sigma$, and $\epsilon$.

---

### Q4 — Attention mechanism (10 points)

Scaled dot-product attention:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**(a)** Why divide by $\sqrt{d_k}$? Show what happens to the softmax gradient if you
omit this scaling and $d_k$ is large.

**(b)** In a protein language model (ESM-2), $Q$, $K$, $V$ all come from the same sequence.
What information does the attention weight $A_{ij} = \text{softmax}(QK^T/\sqrt{d_k})_{ij}$
capture about residues $i$ and $j$?

**(c)** Multi-head attention runs $h$ attention heads in parallel. Write the formula for
multi-head attention with $h$ heads, defining all symbols. Why use multiple heads
rather than one large attention?

---

### Q5 — Practical debugging (10 points)

A colleague shows you the following training curve: their DeepBind CNN has
training loss steadily decreasing to 0.12, but validation AUROC stays at 0.52
(essentially random) throughout all 100 epochs.

**(a)** List three distinct hypotheses for what is wrong. For each, describe
a diagnostic test that would confirm or rule it out.

**(b)** The colleague has accidentally shuffled the labels independently of the
sequences (i.e., labels are now random with respect to sequences). How would
the training and validation curves look different from the scenario above?

**(c)** Write a 5-step debugging protocol for any new biological deep learning
model, starting with overfit-one-batch. For each step, describe what you
are testing and what a "pass" looks like.

---

In [ ]:
# Assessment stub implementations
# Write your implementations in exercises/12_assessment_answers.md
# The functions below raise NotImplementedError — replace with your implementation

def backprop_2layer(X, y, W1, b1, W2, b2, lr=0.01):
    """
    Q1: One step of forward + backward pass for a 2-layer network.
    Returns updated (W1, b1, W2, b2) and the loss value.
    
    Expected:
    - forward: z1 = W1@X + b1, a1 = ReLU(z1), z2 = W2@a1 + b2, yhat = sigmoid(z2)
    - backward: derive all gradients via chain rule
    """
    raise NotImplementedError("Q1: implement 2-layer backprop from scratch")

def compute_normalized_adj(n_nodes, edge_list):
    """
    Q2(c): Compute the GCN normalized adjacency matrix A_norm = D_tilde^{-1/2} A_tilde D_tilde^{-1/2}.
    
    Args:
        n_nodes: int
        edge_list: list of (u, v) tuples (undirected)
    Returns:
        A_norm: np.ndarray (n_nodes, n_nodes)
    """
    raise NotImplementedError("Q2: implement GCN normalized adjacency")

def kl_gaussian(mu, logvar):
    """
    Q3(b): Closed-form KL divergence for N(mu, sigma^2) vs N(0, I).
    Both mu and logvar are torch tensors of shape (batch, latent_dim).
    Returns: scalar KL (sum over latent dims, mean over batch)
    """
    raise NotImplementedError("Q3: implement KL divergence closed form")

def scaled_dot_product_attention_manual(Q, K, V, d_k):
    """
    Q4: Implement scaled dot-product attention from scratch.
    Q, K, V: np.ndarray of shape (seq_len, d_k)
    Returns: output (seq_len, d_k), attention_weights (seq_len, seq_len)
    """
    raise NotImplementedError("Q4: implement scaled dot-product attention")


# ---- Evaluation harness (do not modify) ----
import traceback

def test_backprop():
    np.random.seed(0)
    X = np.random.randn(4, 5)   # 5 samples, 4 features
    y = (np.random.randn(5) > 0).astype(float)
    W1 = np.random.randn(8, 4) * 0.1
    b1 = np.zeros(8)
    W2 = np.random.randn(1, 8) * 0.1
    b2 = np.zeros(1)
    try:
        result = backprop_2layer(X, y, W1, b1, W2, b2)
        assert len(result) == 5, 'Must return (W1, b1, W2, b2, loss)'
        loss = result[4]
        assert 0 < loss < 2, f'Loss {loss:.4f} looks wrong for binary CE'
        print('[Q1] backprop_2layer: PASS (loss in plausible range)')
    except NotImplementedError:
        print('[Q1] backprop_2layer: NOT IMPLEMENTED')
    except Exception as e:
        print(f'[Q1] backprop_2layer: ERROR — {e}')

def test_normalized_adj():
    edge_list = [(0,1),(1,2),(2,3),(0,3)]
    try:
        A_norm = compute_normalized_adj(4, edge_list)
        assert A_norm.shape == (4, 4), 'Shape must be (4,4)'
        # Diagonal should be non-zero (self-loops)
        assert all(A_norm[i, i] > 0 for i in range(4)), 'Diagonal must be > 0 (self-loops)'
        # Matrix should be symmetric
        assert np.allclose(A_norm, A_norm.T, atol=1e-6), 'Must be symmetric'
        print('[Q2] compute_normalized_adj: PASS')
    except NotImplementedError:
        print('[Q2] compute_normalized_adj: NOT IMPLEMENTED')
    except Exception as e:
        print(f'[Q2] compute_normalized_adj: ERROR — {e}')

def test_kl_gaussian():
    import torch
    mu = torch.zeros(16, 10)
    logvar = torch.zeros(16, 10)  # std=1 => KL should be 0
    try:
        kl = kl_gaussian(mu, logvar)
        assert abs(kl.item()) < 1e-4, f'KL for N(0,1) vs N(0,1) should be 0, got {kl.item():.4f}'
        mu2 = torch.ones(16, 10)
        kl2 = kl_gaussian(mu2, logvar)
        # KL(N(1,1)||N(0,1)) = 0.5 per dim
        assert abs(kl2.item() - 5.0) < 0.5, f'Expected ~5.0, got {kl2.item():.4f}'
        print('[Q3] kl_gaussian: PASS')
    except NotImplementedError:
        print('[Q3] kl_gaussian: NOT IMPLEMENTED')
    except Exception as e:
        print(f'[Q3] kl_gaussian: ERROR — {e}')

def test_attention():
    np.random.seed(42)
    L, d = 10, 16
    Q = np.random.randn(L, d)
    K = np.random.randn(L, d)
    V = np.random.randn(L, d)
    try:
        out, attn = scaled_dot_product_attention_manual(Q, K, V, d)
        assert out.shape == (L, d), f'Output shape must be ({L},{d}), got {out.shape}'
        assert attn.shape == (L, L), f'Attn shape must be ({L},{L}), got {attn.shape}'
        # Attention weights must sum to 1 per row
        assert np.allclose(attn.sum(axis=1), 1.0, atol=1e-5), 'Attn rows must sum to 1'
        print('[Q4] scaled_dot_product_attention_manual: PASS')
    except NotImplementedError:
        print('[Q4] scaled_dot_product_attention_manual: NOT IMPLEMENTED')
    except Exception as e:
        print(f'[Q4] scaled_dot_product_attention_manual: ERROR — {e}')

print('=== Assessment stub evaluation ===')
test_backprop()
test_normalized_adj()
test_kl_gaussian()
test_attention()


<!-- REFERENCE IMPLEMENTATIONS (do not read before attempting)

def backprop_2layer(X, y, W1, b1, W2, b2, lr=0.01):
    n = X.shape[1]
    # Forward
    z1 = W1 @ X + b1[:, np.newaxis]
    a1 = np.maximum(0, z1)                            # ReLU
    z2 = W2 @ a1 + b2[:, np.newaxis]
    yhat = 1 / (1 + np.exp(-z2))                      # sigmoid
    loss = -np.mean(y * np.log(yhat + 1e-9) + (1-y) * np.log(1 - yhat + 1e-9))
    # Backward
    delta2 = yhat - y.reshape(1, -1)                  # (1, n)
    dW2 = delta2 @ a1.T / n                           # (1, h)
    db2 = delta2.sum(axis=1) / n                      # (1,)
    delta1 = (W2.T @ delta2) * (z1 > 0)              # ReLU gradient
    dW1 = delta1 @ X.T / n
    db1 = delta1.sum(axis=1) / n
    return W1 - lr*dW1, b1 - lr*db1, W2 - lr*dW2, b2 - lr*db2, loss

def compute_normalized_adj(n_nodes, edge_list):
    A = np.zeros((n_nodes, n_nodes), dtype=np.float32)
    for u, v in edge_list:
        A[u, v] = A[v, u] = 1.0
    np.fill_diagonal(A, 1.0)                          # self-loops => A_tilde
    D = A.sum(axis=1)
    D_inv_sqrt = 1.0 / np.sqrt(np.maximum(D, 1e-8))
    return D_inv_sqrt[:, np.newaxis] * A * D_inv_sqrt[np.newaxis, :]

def kl_gaussian(mu, logvar):
    import torch
    # KL(N(mu,sigma^2) || N(0,I)) = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.shape[0]

def scaled_dot_product_attention_manual(Q, K, V, d_k):
    scores = Q @ K.T / np.sqrt(d_k)                   # (L, L)
    scores -= scores.max(axis=1, keepdims=True)        # numerical stability
    attn = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)
    return attn @ V, attn                              # (L, d_k), (L, L)

-->

---
## Module 14 — Completion Checklist

- [ ] NB01: Backpropagation from scratch — can derive all gradients unprompted
- [ ] NB02: PyTorch autograd and nn.Module — can write a custom `Module` class
- [ ] NB03: DeepBind CNN — can explain global max pool and convolutional motif detection
- [ ] NB04: LSTM — can write out all four gate equations from memory
- [ ] NB05: Transformer — can derive scaled dot-product attention and explain sinusoidal PE
- [ ] NB06: VAE — can derive ELBO and reparameterization trick
- [ ] NB07: GCN — can write the normalized adjacency computation from scratch
- [ ] NB08: Molecular GNN — can explain message passing and global pooling
- [ ] NB09: Training best practices — know the 5-step debugging protocol
- [ ] NB10: Interpretability — can implement saliency and integrated gradients
- [ ] NB11: AlphaFold + ESM — can explain the 4-step AF2 pipeline and pLDDT
- [ ] NB12 (this notebook): MP01 complete, all 5 assessment questions answered
- [ ] All exercises in `exercises/` attempted before reading reference implementations

---
*Module 14 complete. Proceed to Module 15: Simulation and Agent-Based Modeling.*